# CampusAid policy Q&A — evaluator notebook

This notebook is a public evaluator for the repository policy assistant. It uses the checked-in Bennett University document corpus and CampusAid’s real `policy_qa` retrieval code; it does not hardcode answers.

- The **retrieval benchmark** runs without a Gemini key and checks ranked source retrieval plus evidence phrases against 12 fixed cases.
- The **saved end-to-end run** below records the actual Gemini-backed output from 2026-11-24. It had 1 pass, 5 failures, and 2 cases that the available sources could not test. Those results remain unchanged.
- Use the interactive question cell to try a new question. It asks for a Gemini key privately and does not save it in this notebook.
- This notebook covers policy Q&A only; it does not host the full grievance submission/tracking interface.

Notebook link: [Open in Google Colab](https://colab.research.google.com/github/akritah/CampusAid/blob/codex/campusaid-evaluator/code/CampusAid_Evaluator_Demo.ipynb).

The attendance section is transcribed from Bennett University’s [2024–25 Examination Manual](https://www.bennett.edu.in/wp-content/uploads/2025/01/Examination-Manual-2024-25-.pdf). Confirm current-year policies with the university.


In [ ]:
# Install only the light dependencies used by the policy QA module (not Torch/Whisper).
%pip -q install fastapi pypdf requests

from pathlib import Path
import os, subprocess, sys, getpass, json

REPO = Path('/content/CampusAid')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'codex/campusaid-evaluator',
        'https://github.com/akritah/CampusAid.git', str(REPO)
    ], check=True)
sys.path.insert(0, str(REPO / 'backend'))
os.environ['POLICY_STORAGE_DIR'] = str(REPO / 'backend' / 'policy_documents')
from app import policy_qa
policy_qa.POLICY_DIR = Path(os.environ['POLICY_STORAGE_DIR'])


def ask_campusaid(question):
    # Input is hidden and held only in this Colab runtime; it is not written to disk.
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key (hidden): ')
    os.environ.setdefault('GEMINI_MODEL', 'gemini-3.5-flash-lite')
    matches = policy_qa.retrieve(question)
    chunks = []
    for i, (chunk, score) in enumerate(matches, 1):
        first_line = chunk.text.splitlines()[0] if chunk.text.splitlines() else ''
        chunks.append({
            'chunk_id': f'C{i}', 'document': chunk.source,
            'section_or_heading': first_line[:160],
            'relevance': round(score, 3), 'text': chunk.text
        })
    response = policy_qa.answer(question)
    return {'input': question, 'retrieved_chunks': chunks, 'actual_output_json': response}


## Fixed retrieval benchmark (no API key required)

This measures retrieval, not generated-answer correctness. Each supported case requires the expected document and specified evidence phrase(s) to appear in the top four retrieved chunks. The unsupported fee-deadline case passes only when no chunks are returned. The set is small and intentionally limited to readable, indexed policy text; it is not a broad or independent benchmark.


In [1]:
import json
from IPython.display import display, Markdown

retrieval_cases = json.loads((REPO / 'test-data' / 'retrieval_cases.json').read_text(encoding='utf-8-sig'))
retrieval_results = []
for case in retrieval_cases:
    matches = policy_qa.retrieve(case['question'], limit=4)
    expected_source = case['expected_source']
    if expected_source:
        relevant = [(chunk, score, rank) for rank, (chunk, score) in enumerate(matches, 1)
                    if chunk.source == expected_source]
        relevant_text = ' '.join(chunk.text.lower() for chunk, _, _ in relevant)
        evidence_found = all(phrase.lower() in relevant_text for phrase in case['evidence'])
        passed = bool(relevant) and evidence_found
        rank = relevant[0][2] if relevant else None
    else:
        relevant = []
        evidence_found = True
        passed = len(matches) == 0
        rank = None
    retrieval_results.append({
        'id': case['id'], 'question': case['question'], 'passed': passed,
        'expected_source': expected_source, 'top_source_rank': rank,
        'evidence_found': evidence_found,
        'retrieved': [{'document': c.source, 'score': round(s, 3), 'text': c.text}
                      for c, s in matches],
    })

supported = [r for r in retrieval_results if r['expected_source']]
hits_at_1 = sum(r['top_source_rank'] == 1 for r in supported)
hits_at_4 = sum(r['top_source_rank'] is not None for r in supported)
rr = [1 / r['top_source_rank'] if r['top_source_rank'] else 0 for r in supported]
false_positive = sum((not r['expected_source']) and bool(r['retrieved']) for r in retrieval_results)
rows = ['| ID | Result | Expected source rank | Retrieved top source |', '|---|---|---:|---|']
for r in retrieval_results:
    top = r['retrieved'][0]['document'] if r['retrieved'] else 'None'
    rows.append(f"| {r['id']} | {'PASS' if r['passed'] else 'FAIL'} | {r['top_source_rank'] or 'n/a'} | {top} |")
display(Markdown('\n'.join(rows)))
print(f"Retrieval checks: {sum(r['passed'] for r in retrieval_results)}/{len(retrieval_results)} passed")
print(f"Supported-query source recall@1: {hits_at_1}/{len(supported)} ({hits_at_1/len(supported):.1%})")
print(f"Supported-query source recall@4: {hits_at_4}/{len(supported)} ({hits_at_4/len(supported):.1%})")
print(f"Supported-query MRR@4: {sum(rr)/len(rr):.3f}")
print(f"Unsupported-query false positives: {false_positive}/{len(retrieval_results)-len(supported)}")
if any(not r['passed'] for r in retrieval_results):
    print('Failed details:', json.dumps([r for r in retrieval_results if not r['passed']], ensure_ascii=False, indent=2))
assert all(r['passed'] for r in retrieval_results), 'At least one fixed retrieval check failed; inspect details above.'


| ID | Result | Expected source rank | Retrieved top source |
|---|---|---:|---|
| R1 | PASS | 1 | Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt |
| R2 | PASS | 1 | Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt |
| R3 | PASS | 1 | Bennett_Grievance_Redressal_Policy.pdf |
| R4 | PASS | 1 | Bennett_Grievance_Redressal_Policy.pdf |
| R5 | PASS | 1 | Bennett_Grievance_Redressal_Policy.pdf |
| R6 | PASS | 1 | Bennett_Grievance_Redressal_Policy.pdf |
| R7 | PASS | 1 | Bennett_Hostel_Policy_Amendments_AY2026-27.pdf |
| R8 | PASS | 1 | Bennett_Hostel_Policy_Amendments_AY2026-27.pdf |
| R9 | PASS | 1 | Bennett_Hostel_Policy_Amendments_AY2026-27.pdf |
| R10 | PASS | 1 | Bennett_Hostel_Policy_Amendments_AY2026-27.pdf |
| R11 | PASS | 1 | Bennett_Hostel_Policy_Amendments_AY2026-27.pdf |
| R12 | PASS | n/a | None |

Retrieval checks: 12/12 passed
Supported-query source recall@1: 11/11 (100.0%)
Supported-query source recall@4: 11/11 (100.0%)
Supported-query MRR@4: 1.000
Unsupported-query false positives: 0/1


## Saved actual run

This is a preserved response from an actual local run on 2026-11-24. It does not call Gemini when the cell is opened or re-run; the JSON run artifact in the repository is the source of these outputs. Review failed and not-testable cases as recorded.


In [ ]:
from IPython.display import display, Markdown
run_path = REPO / 'test-data' / 'policy_qa_run_2026-11-24.json'
SAVED_SAMPLE = json.loads(run_path.read_text(encoding='utf-8'))['results'][0]
SAVED_SAMPLE = {'input': SAVED_SAMPLE['input'], 'retrieved_chunks': SAVED_SAMPLE['retrieved_chunks'], 'actual_output_json': SAVED_SAMPLE['actual_output_json']}
saved_run = json.loads(run_path.read_text(encoding='utf-8'))
rows = ['| Test | Result | Retrieved documents | Actual answer (short) |', '|---|---|---|---|']
for item in saved_run['results']:
    docs = ', '.join(c['document'] for c in item['retrieved_chunks']) or 'None'
    short = item['actual_output_json'].get('answer', 'HTTP ' + str(item['actual_output_json'].get('http_status'))).replace('|', r'\|')
    rows.append(f"| {item['id']} | {item['observed_status']} | {docs} | {short} |")
display(Markdown('\n'.join(rows)))
print('Sample run — exact recorded input, retrieved chunks and CampusAid JSON:')
print(json.dumps(SAVED_SAMPLE, ensure_ascii=False, indent=2))


CampusAid policy Q&A — recorded live evaluation (2026-11-24)
Run: backend/app/policy_qa.py + indexed repository documents + configured Gemini API. Key excluded.
Summary: 1 pass, 5 fail, 2 not testable.

T1 [PASS] What is the minimum attendance required to sit for end-semester exams?
Retrieved: Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt
Actual: The minimum attendance required to sit for end-semester examinations is 75% in each course during a semester [Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt].

T2 [FAIL] How do I apply for re-evaluation of my exam paper?
Retrieved: Bennett_Examination_Manual_AY2024-25_Attendance_Section.txt
Actual: The documents do not specify how to apply for the re-evaluation of an exam paper. Please contact the relevant examination office for this information.

T3 [FAIL] What is the procedure for taking leave from the hostel?
Retrieved: no chunk
Actual: To take leave from a university hostel, students typically need to submit a fo

## Try a new question

Run this cell after setup to query the same CampusAid retriever and answer function with a question of your choice. A Gemini API key is required for new generated answers; it is prompted privately. No key is stored in the notebook.


In [ ]:
question = input('Ask a university policy question: ')
result = ask_campusaid(question)
print(json.dumps(result, ensure_ascii=False, indent=2))
